# Fabric Defect YOLOv8-cls Training Notebook
Run cells top-to-bottom on Google Colab (GPU runtime).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install ultralytics -q
import ultralytics
ultralytics.checks()  # confirms GPU is detected


In [ ]:
import torch
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

TILDA_ROOT = Path("/content/drive/MyDrive/tilda_CD1+CD2")
BASE       = Path("/content/drive/MyDrive/fabric_defect_yolov8")

assert TILDA_ROOT.exists(), f"Cannot find {TILDA_ROOT} — check your Drive path"
print(f"TILDA root found: {TILDA_ROOT}")


## Step 1 — Audit TILDA text files to confirm class mapping

In [ ]:
from pathlib import Path

sample_text_root = TILDA_ROOT / "cd2" / "c4" / "r1" / "text"

for e_folder in sorted(sample_text_root.iterdir()):
    if not e_folder.is_dir():
        continue
    files = sorted(e_folder.iterdir())
    if not files:
        print(f"
{e_folder.name}/ → EMPTY")
        continue
    print(f"
{'─'*40}")
    print(f"Folder: {e_folder.name}/   ({len(files)} files total)")
    for f in files[:2]:
        try:
            content = f.read_text(encoding="latin-1", errors="ignore").strip()
            print(f"  [{f.name}]: {content}")
        except Exception as ex:
            print(f"  [{f.name}]: ERROR — {ex}")


## Step 2 — Build classification dataset (70/15/15 stratified split)

In [ ]:
import shutil, cv2, numpy as np, random
from pathlib import Path
from collections import defaultdict
from PIL import Image

random.seed(42)

E_CLASS_MAP = {
    "e0": None,          # background — excluded
    "e1": "Hole",
    "e2": "Stain",
    "e3": "Broken_Thread",
    "e4": "Misweave",
    "e5": None,          # crease (handling artifact) — excluded
    "e6": None,          # lamp-off calibration — excluded
    "e7": None,          # camera-distance calibration — excluded
}

CLASS_NAMES = ["Hole", "Stain", "Broken_Thread", "Misweave"]

# ── Create output folder structure ────────────────────────────
cls_dataset = BASE / "dataset_cls"
if cls_dataset.exists():
    shutil.rmtree(cls_dataset)
for split in ["train", "val", "test"]:
    for cls in CLASS_NAMES:
        (cls_dataset / split / cls).mkdir(parents=True)
print("Classification dataset folders created.")

# ── Collect all defect images by class ────────────────────────
defects = defaultdict(list)
for cd_folder in sorted(TILDA_ROOT.iterdir()):
    if not cd_folder.is_dir(): continue
    for cloth_folder in sorted(cd_folder.iterdir()):
        if not cloth_folder.is_dir(): continue
        for rep_folder in sorted(cloth_folder.iterdir()):
            if not rep_folder.is_dir(): continue
            images_root = rep_folder / "images"
            if not images_root.exists(): continue
            for e_folder in sorted(images_root.iterdir()):
                if not e_folder.is_dir(): continue
                cls = E_CLASS_MAP.get(e_folder.name)
                if cls not in CLASS_NAMES: continue
                for ext in ["*.TIF", "*.tif"]:
                    defects[cls].extend(e_folder.glob(ext))

print("Images found per class:")
for cls, imgs in defects.items():
    print(f"  {cls:<18}: {len(imgs)}")

# ── FIX: complete load_and_convert with CLAHE ─────────────────
# (the original notebook cell was truncated mid-function)
CLAHE_ENGINE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_and_convert(tif_path):
    """Load TIF → greyscale → CLAHE → 3-ch RGB → letterbox 640×640."""
    img  = Image.open(tif_path).convert("RGB")
    arr  = np.array(img)
    # Convert to LAB, apply CLAHE on L channel, convert back
    lab  = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = CLAHE_ENGINE.apply(lab[:, :, 0])
    rgb  = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    # Letterbox to 640×640
    h, w = rgb.shape[:2]
    scale  = 640 / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas  = np.full((640, 640, 3), 114, dtype=np.uint8)
    pad_top  = (640 - new_h) // 2
    pad_left = (640 - new_w) // 2
    canvas[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized
    return canvas  # RGB uint8

# ── Stratified split and copy ──────────────────────────────────
SPLIT_RATIOS = (0.70, 0.15, 0.15)
copied = defaultdict(int)

for cls, paths in defects.items():
    random.shuffle(paths)
    n     = len(paths)
    n_tr  = int(n * SPLIT_RATIOS[0])
    n_val = int(n * SPLIT_RATIOS[1])
    splits = {
        "train": paths[:n_tr],
        "val":   paths[n_tr:n_tr+n_val],
        "test":  paths[n_tr+n_val:],
    }
    for split, split_paths in splits.items():
        for i, src in enumerate(split_paths):
            out = cls_dataset / split / cls / f"{cls}_{split}_{i:04d}.jpg"
            try:
                img = load_and_convert(src)
                Image.fromarray(img).save(out, "JPEG", quality=95)
                copied[split] += 1
            except Exception as e:
                print(f"  SKIP {src.name}: {e}")

print("
Images copied per split:")
for split, n in copied.items():
    print(f"  {split}: {n}")


## Step 3 — Train YOLOv8m-cls

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m-cls.pt")

results = model.train(
    data         = str(cls_dataset),
    epochs       = 50,
    imgsz        = 224,
    batch        = 32,
    lr0          = 1e-3,
    lrf          = 1e-5,
    cos_lr       = True,
    momentum     = 0.937,
    weight_decay = 5e-4,
    warmup_epochs= 3,
    patience     = 20,
    flipud       = 0.5,
    fliplr       = 0.5,
    degrees      = 10.0,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    device       = 0,
    project      = str(BASE / "runs"),
    name         = "yolov8_cls_tilda",
    exist_ok     = True,
    plots        = True,
    save         = True,
)
print(f"
Best weights: {results.save_dir}/weights/best.pt")


## Step 4 — Evaluate on test split
**FIX**: original Cell 8 used  (detection API).
Classification models expose  / .

In [ ]:
from ultralytics import YOLO

best = YOLO(str(BASE / "runs/yolov8_cls_tilda/weights/best.pt"))

metrics = best.val(
    data   = str(cls_dataset),
    split  = "test",
    imgsz  = 224,
    batch  = 32,
    device = 0,
)

# ── FIX: use classification metrics, NOT metrics.box ──────────
# Old (wrong) code used: metrics.box.mp, metrics.box.mr, metrics.box.map
# Those attributes belong to detection tasks and raise AttributeError here.
print(f"
{chr(61)*45}")
print(f"Top-1 Accuracy : {metrics.top1:.4f}")
print(f"Top-5 Accuracy : {metrics.top5:.4f}")
print(f"{chr(61)*45}")

# Per-class breakdown from confusion matrix
names = ["Hole", "Stain", "Broken_Thread", "Misweave"]
print("
Per-class results (from confusion matrix):")
try:
    cm = metrics.confusion_matrix.matrix
    import numpy as np
    for i, name in enumerate(names):
        tp = cm[i, i]
        total = cm[:, i].sum()
        acc = tp / total if total > 0 else 0
        print(f"  {name:<18}: {acc:.3f}")
except Exception:
    print("  (confusion matrix not available in this ultralytics version)")


## Step 5 — Download best weights

In [ ]:
from google.colab import files
files.download(str(BASE / "runs/yolov8_cls_tilda/weights/best.pt"))
# Place the downloaded best.pt next to app.py, model.py, analyzer.py
